In [0]:
%pip install torch-sparse
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CUDA_ALLOC_CONF must be set before any CUDA context exists (before
# `import torch`) -- fixes a real fragmentation OOM on deeper K=2/K=3
# NeighborLoader batches, even after earlier epochs succeeded.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## Setup

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import degree

from training_data import Ontology, TrainingData
from node_prediction.gnn_tools import (
    parse_neighbors,
    get_graph_mask,
    train_final_model,
    evaluate_test_set,
    compute_entity_reuse_stats,
    avg_children_per_case,
    train_model,
    evaluate_ablation_configuration,
    run_ablation_run,
    run_sequential_optimization_with_verification,
    run_k_sweep,
)

In [0]:
FEATURE_DIM = 1024
HIDDEN_DIM = 64
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"

sulo_schema_path = f"{volume}sulo_kg.ttl"
sulo_data_path = f"{volume}subgraph_balanced_sulo_populated_graph.nt"
sulo_vars = f'{volume}sulo_vars.pt'

baseline_schema_path = f"{volume}base_kg.ttl"
baseline_data_path = f"{volume}subset_balanced_baseline_populated_graph.nt"
baseline_vars = f'{volume}base_vars.pt'

## Create Training Data and Graph

In [0]:
def cap_max_degree(edge_index, edge_type, num_nodes, max_degree=128, seed=42):
    """
    Caps in-degree per node at max_degree by randomly keeping at most
    max_degree incoming edges for any node whose in-degree exceeds it.
    Applied to the combined (already-bidirectional) edge_index, so this
    bounds fan-in from both directions -- exactly what NeighborLoader's
    per-hop sampling actually pulls from.
    """
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(edge_index.size(1), generator=g)
    edge_index = edge_index[:, perm]
    edge_type = edge_type[perm]

    dst = edge_index[1]
    order = torch.argsort(dst, stable=True)  # stable sort preserves the random shuffle within each dst group
    dst_sorted = dst[order]

    counts = torch.bincount(dst_sorted, minlength=num_nodes)
    group_start = torch.zeros(num_nodes, dtype=torch.long)
    group_start[1:] = torch.cumsum(counts[:-1], dim=0)
    pos_in_group = torch.arange(dst_sorted.size(0)) - group_start[dst_sorted]

    keep_mask_sorted = pos_in_group < max_degree
    keep_order = order[keep_mask_sorted]

    print(f"Capped edges: {edge_index.size(1)} -> {keep_order.size(0)} "
          f"({edge_index.size(1) - keep_order.size(0)} dropped, max_degree={max_degree})")
    return edge_index[:, keep_order], edge_type[keep_order]

def compute_degree_feature(edge_index, num_nodes):
    """Log-scaled, z-normalized in-degree -- lets the model tell hub nodes
    apart even when they share an identical categorical fallback vector."""
    deg = degree(edge_index[1], num_nodes=num_nodes).float()
    log_deg = torch.log1p(deg)
    normalized = (log_deg - log_deg.mean()) / (log_deg.std() + 1e-8)
    return normalized

## Intialize Model and Training

In [0]:
class RGCNModel(nn.Module):
    def __init__(self, in_channels, hidden_channels_list, num_relations, 
                 vector_table, categorical_row_ids, num_bases=None, 
                 out_channels=1, dropout=0.2):
        super().__init__()
        num_rows, emb_dim = vector_table.shape

        is_categorical = torch.zeros(num_rows, dtype=torch.bool)
        is_categorical[categorical_row_ids] = True
        self.register_buffer('is_categorical_mask', is_categorical)

        # Plain attribute, not a registered buffer -- vector_table is a
        # multi-GB shared table; registering it would clone a fresh copy on
        # every model_factory() call (construction + .to()/.cpu()), which
        # OOM'd the sweep after a few configs. Caller must move it to the
        # target device once, outside the sweep loop.
        self.frozen_table = vector_table

        self.categorical_embedding = nn.Embedding(len(categorical_row_ids), emb_dim)
        with torch.no_grad():
            self.categorical_embedding.weight.copy_(
                vector_table[categorical_row_ids.to(vector_table.device)]
            )

        local_id = torch.full((num_rows,), -1, dtype=torch.long)
        local_id[categorical_row_ids] = torch.arange(len(categorical_row_ids))
        self.register_buffer('local_categorical_id', local_id)

        self.convs = nn.ModuleList()
        dims = [in_channels] + hidden_channels_list
        for i in range(len(dims) - 1):
            self.convs.append(RGCNConv(dims[i], dims[i + 1], num_relations=num_relations, num_bases=num_bases))
        self.convs.append(RGCNConv(dims[-1], out_channels, num_relations=num_relations, num_bases=num_bases))
        self.dropout = dropout

    def forward(self, node_row_ids, edge_index, edge_type, degree_feat):
        x = self.frozen_table[node_row_ids].clone()

        cat_mask = self.is_categorical_mask[node_row_ids]
        if cat_mask.any():
            local_ids = self.local_categorical_id[node_row_ids[cat_mask]]
            x[cat_mask] = self.categorical_embedding(local_ids)

        x = torch.cat([x, degree_feat.unsqueeze(1)], dim=1)

        for conv in self.convs[:-1]:
            x = conv(x, edge_index, edge_type)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index, edge_type)
        return x

## Hyperparameter Tuning

In [0]:
# ── 1. CONFIGURATION & DIRECTORIES ───────────────────────────────────────────
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
os.makedirs(f"{volume}rgcn/tuning_logs/", exist_ok=True)

EPOCHS = 50
PATIENCE = 15
DEFAULT_BASES = 30
DEFAULT_LR = 1e-3
DEFAULT_WD = 1e-4

In [ ]:
schema_label = 'base'

csv_path = f"{volume}rgcn/{schema_label}_sequential_tuning_results.csv"
ablation_csv_path = f"{volume}rgcn/{schema_label}_ablation_study_results.csv"

# ── 2. RETRIEVE METRIC & ARCHITECTURE WINNERS FROM SEQUENTIAL SWEEP ──────────
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Could not find tuning CSV results at: {csv_path}")

df_tuning = pd.read_csv(csv_path)

phase3_runs = df_tuning[(df_tuning['status'] == 'ok') & (df_tuning['phase'].isin(['3_Verification', '3_Verification_Cached']))]
if len(phase3_runs) == 0:
    phase3_runs = df_tuning[df_tuning['status'] == 'ok']
best_row = phase3_runs.loc[phase3_runs['val_roc_auc'].idxmax()]

optimal_dim = int(best_row['hidden_dim'])
optimal_bases = best_row['num_bases']
optimal_bases = None if pd.isna(optimal_bases) or optimal_bases == 'None' else int(float(optimal_bases))
optimal_lr = float(best_row['lr'])
optimal_wd = float(best_row['weight_decay'])
best_depth = int(best_row['layers'])
optimal_neighbors = parse_neighbors(best_row['neighbors'])

print(f">>> Dynamic Hyperparameter Handoff Complete for {schema_label.upper()}:")
print(f"  * Hidden Dim: {optimal_dim} | Depth: {best_depth} | Bases: {optimal_bases} | LR: {optimal_lr} | WD: {optimal_wd}")

# ── 3. SELF-HEALING ONTOLOGY LOOKUP HELPERS ──────────────────────────────────
vars_path = f"{volume}{schema_label}_vars.pt"
ckpt = torch.load(vars_path, map_location='cpu', weights_only=False)

graph_data = ckpt[f'{schema_label}_graph_data']
train_indices = ckpt['x_train']
val_indices = ckpt['x_val']
test_indices = ckpt['x_test']
vector_table = ckpt[f'{schema_label}_vector_table']

# Moved once, outside the ablation/K-sweep loops -- see RGCNModel's comment.
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
vector_table = vector_table.to(_device)

num_relations = graph_data.edge_type.max().item() + 1
node_index = graph_data.n_id_to_vec_row
usage_counts = torch.bincount(node_index, minlength=vector_table.shape[0])

# Cache-first: avoids raw .nt access once {schema_label}_ontology_enums.pt exists.
_enum_cache_path = f"{volume}{schema_label}_ontology_enums.pt"
if os.path.exists(_enum_cache_path):
    from types import SimpleNamespace
    _enum_cache = torch.load(_enum_cache_path, map_location='cpu', weights_only=False)
    ontology_obj = SimpleNamespace(
        enum_nodes=_enum_cache['enum_nodes'],
        enum_relations=_enum_cache['enum_relations'],
    )
    print(f"[{schema_label}] Loaded cached enum_nodes/enum_relations from {_enum_cache_path} "
          f"-- no .nt file access needed.")
elif schema_label == 'sulo':
    ontology_obj = Ontology(nt_file_path=sulo_data_path, label_predicate="hasFeature", label_object_pattern=r"SurveyRating/(\d+)")
else:
    ontology_obj = Ontology(nt_file_path=baseline_data_path, label_predicate="hasSurveyRating", label_object_pattern=r"SurveyRating/(\d+)")

## K Sweep Test

In [0]:
# ── EXPERIMENT 2: K-HOP DEPTH SWEEP (RQ1b) ────────────────────────────────────
print("\n>>> STARTING K-SWEEP: STRUCTURAL SENSITIVITY (K = 1, 2, 3)")

# Everything except K (depth/fanout) is held at the already-tuned winning
# config, so any metric difference across rows is attributable to K alone.
def k_sweep_model_factory(hidden_list, bases):
    return RGCNModel(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=hidden_list,
        num_relations=num_relations,
        vector_table=vector_table,
        categorical_row_ids=torch.where(usage_counts > 1)[0],
        num_bases=bases,
    )

# Fan-out fixed per graph at its measured 99th-percentile neighbourhood
# size, held constant across K=1,2,3 to isolate depth alone (unlike the
# GRID_1B_DEPTH schedule above, which varies fan-out per depth instead).
_rq1a_csv_path = f"{volume}graph_geometry/rq1a_structural_comparison.csv"
if not os.path.exists(_rq1a_csv_path):
    raise FileNotFoundError(
        f"Could not find Table 1's structural comparison CSV at: {_rq1a_csv_path}. "
        f"Run extract_graph_geometry.ipynb first -- the K-sweep's fixed fan-out is "
        f"derived from its p99_degree_threshold measurement."
    )
_rq1a_df = pd.read_csv(_rq1a_csv_path, index_col='metric')
_kg_display = {'base': 'Ad-hoc (Base)', 'sulo': 'SULO'}[schema_label]
_p99_fanout = int(round(float(_rq1a_df.loc['p99_degree_threshold', _kg_display])))
print(f">>> Fixed fan-out for {_kg_display}: {_p99_fanout} "
      f"(99th-percentile neighbourhood size, from rq1a_structural_comparison.csv)")

K_SWEEP_VALUES = [
    (1, (_p99_fanout,)),
    (2, (_p99_fanout, _p99_fanout)),
    (3, (_p99_fanout, _p99_fanout, _p99_fanout)),
]

k_sweep_csv_path = f"{volume}rgcn/{schema_label}_k_sweep_results.csv"

k_sweep_df = run_k_sweep(
    model_factory=k_sweep_model_factory,
    graph_data=graph_data, schema_label=schema_label,
    train_indices=train_indices, val_indices=val_indices, test_indices=test_indices,
    vector_table=vector_table, usage_counts=usage_counts,
    hidden_dim=optimal_dim, bases=optimal_bases, lr=optimal_lr, wd=optimal_wd,
    k_values=K_SWEEP_VALUES,
    epochs=50, patience=15,
    csv_path=k_sweep_csv_path,
    resume=True,
)

print(f"\n=======================================================")
print(f" K-SWEEP COMPLETE FOR {schema_label.upper()}")
print(f"=======================================================")
print(k_sweep_df.to_string(index=False))
print(f"=======================================================")

## Rerun


In [0]:
%pip install torch-sparse
%restart_python

In [0]:
# CUDA_ALLOC_CONF must be set before any CUDA context exists (before
# `import torch`) -- fixes a real fragmentation OOM on deeper K=2/K=3
# NeighborLoader batches, even after earlier epochs succeeded.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## Setup

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import degree

from training_data import Ontology, TrainingData
from node_prediction.gnn_tools import (
    parse_neighbors,
    get_graph_mask,
    train_final_model,
    evaluate_test_set,
    compute_entity_reuse_stats,
    avg_children_per_case,
    train_model,
    evaluate_ablation_configuration,
    run_ablation_run,
    run_sequential_optimization_with_verification,
    run_k_sweep,
)

In [0]:
FEATURE_DIM = 1024
HIDDEN_DIM = 64
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"

sulo_schema_path = f"{volume}sulo_kg.ttl"
sulo_data_path = f"{volume}subgraph_balanced_sulo_populated_graph.nt"
sulo_vars = f'{volume}sulo_vars.pt'

baseline_schema_path = f"{volume}base_kg.ttl"
baseline_data_path = f"{volume}subset_balanced_baseline_populated_graph.nt"
baseline_vars = f'{volume}base_vars.pt'

## Create Training Data and Graph

In [0]:
def cap_max_degree(edge_index, edge_type, num_nodes, max_degree=128, seed=42):
    """
    Caps in-degree per node at max_degree by randomly keeping at most
    max_degree incoming edges for any node whose in-degree exceeds it.
    Applied to the combined (already-bidirectional) edge_index, so this
    bounds fan-in from both directions -- exactly what NeighborLoader's
    per-hop sampling actually pulls from.
    """
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(edge_index.size(1), generator=g)
    edge_index = edge_index[:, perm]
    edge_type = edge_type[perm]

    dst = edge_index[1]
    order = torch.argsort(dst, stable=True)  # stable sort preserves the random shuffle within each dst group
    dst_sorted = dst[order]

    counts = torch.bincount(dst_sorted, minlength=num_nodes)
    group_start = torch.zeros(num_nodes, dtype=torch.long)
    group_start[1:] = torch.cumsum(counts[:-1], dim=0)
    pos_in_group = torch.arange(dst_sorted.size(0)) - group_start[dst_sorted]

    keep_mask_sorted = pos_in_group < max_degree
    keep_order = order[keep_mask_sorted]

    print(f"Capped edges: {edge_index.size(1)} -> {keep_order.size(0)} "
          f"({edge_index.size(1) - keep_order.size(0)} dropped, max_degree={max_degree})")
    return edge_index[:, keep_order], edge_type[keep_order]

def compute_degree_feature(edge_index, num_nodes):
    """Log-scaled, z-normalized in-degree -- lets the model tell hub nodes
    apart even when they share an identical categorical fallback vector."""
    deg = degree(edge_index[1], num_nodes=num_nodes).float()
    log_deg = torch.log1p(deg)
    normalized = (log_deg - log_deg.mean()) / (log_deg.std() + 1e-8)
    return normalized

## Intialize Model and Training

In [0]:
class RGCNModel(nn.Module):
    def __init__(self, in_channels, hidden_channels_list, num_relations, 
                 vector_table, categorical_row_ids, num_bases=None, 
                 out_channels=1, dropout=0.2):
        super().__init__()
        num_rows, emb_dim = vector_table.shape

        is_categorical = torch.zeros(num_rows, dtype=torch.bool)
        is_categorical[categorical_row_ids] = True
        self.register_buffer('is_categorical_mask', is_categorical)

        # Plain attribute, not a registered buffer -- vector_table is a
        # multi-GB shared table; registering it would clone a fresh copy on
        # every model_factory() call (construction + .to()/.cpu()), which
        # OOM'd the sweep after a few configs. Caller must move it to the
        # target device once, outside the sweep loop.
        self.frozen_table = vector_table

        self.categorical_embedding = nn.Embedding(len(categorical_row_ids), emb_dim)
        with torch.no_grad():
            self.categorical_embedding.weight.copy_(
                vector_table[categorical_row_ids.to(vector_table.device)]
            )

        local_id = torch.full((num_rows,), -1, dtype=torch.long)
        local_id[categorical_row_ids] = torch.arange(len(categorical_row_ids))
        self.register_buffer('local_categorical_id', local_id)

        self.convs = nn.ModuleList()
        dims = [in_channels] + hidden_channels_list
        for i in range(len(dims) - 1):
            self.convs.append(RGCNConv(dims[i], dims[i + 1], num_relations=num_relations, num_bases=num_bases))
        self.convs.append(RGCNConv(dims[-1], out_channels, num_relations=num_relations, num_bases=num_bases))
        self.dropout = dropout

    def forward(self, node_row_ids, edge_index, edge_type, degree_feat):
        x = self.frozen_table[node_row_ids].clone()

        cat_mask = self.is_categorical_mask[node_row_ids]
        if cat_mask.any():
            local_ids = self.local_categorical_id[node_row_ids[cat_mask]]
            x[cat_mask] = self.categorical_embedding(local_ids)

        x = torch.cat([x, degree_feat.unsqueeze(1)], dim=1)

        for conv in self.convs[:-1]:
            x = conv(x, edge_index, edge_type)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index, edge_type)
        return x

## Hyperparameter Tuning

In [0]:
# ── 1. CONFIGURATION & DIRECTORIES ───────────────────────────────────────────
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
os.makedirs(f"{volume}rgcn/tuning_logs/", exist_ok=True)

EPOCHS = 50
PATIENCE = 15
DEFAULT_BASES = 30
DEFAULT_LR = 1e-3
DEFAULT_WD = 1e-4

In [ ]:
schema_label = 'sulo'

csv_path = f"{volume}rgcn/{schema_label}_sequential_tuning_results.csv"
ablation_csv_path = f"{volume}rgcn/{schema_label}_ablation_study_results.csv"

# ── 2. RETRIEVE METRIC & ARCHITECTURE WINNERS FROM SEQUENTIAL SWEEP ──────────
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Could not find tuning CSV results at: {csv_path}")

df_tuning = pd.read_csv(csv_path)

phase3_runs = df_tuning[(df_tuning['status'] == 'ok') & (df_tuning['phase'].isin(['3_Verification', '3_Verification_Cached']))]
if len(phase3_runs) == 0:
    phase3_runs = df_tuning[df_tuning['status'] == 'ok']
best_row = phase3_runs.loc[phase3_runs['val_roc_auc'].idxmax()]

optimal_dim = int(best_row['hidden_dim'])
optimal_bases = best_row['num_bases']
optimal_bases = None if pd.isna(optimal_bases) or optimal_bases == 'None' else int(float(optimal_bases))
optimal_lr = float(best_row['lr'])
optimal_wd = float(best_row['weight_decay'])
best_depth = int(best_row['layers'])
optimal_neighbors = parse_neighbors(best_row['neighbors'])

print(f">>> Dynamic Hyperparameter Handoff Complete for {schema_label.upper()}:")
print(f"  * Hidden Dim: {optimal_dim} | Depth: {best_depth} | Bases: {optimal_bases} | LR: {optimal_lr} | WD: {optimal_wd}")

# ── 3. SELF-HEALING ONTOLOGY LOOKUP HELPERS ──────────────────────────────────
vars_path = f"{volume}{schema_label}_vars.pt"
ckpt = torch.load(vars_path, map_location='cpu', weights_only=False)

graph_data = ckpt[f'{schema_label}_graph_data']
train_indices = ckpt['x_train']
val_indices = ckpt['x_val']
test_indices = ckpt['x_test']
vector_table = ckpt[f'{schema_label}_vector_table']

# Moved once, outside the ablation/K-sweep loops -- see RGCNModel's comment.
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
vector_table = vector_table.to(_device)

num_relations = graph_data.edge_type.max().item() + 1
node_index = graph_data.n_id_to_vec_row
usage_counts = torch.bincount(node_index, minlength=vector_table.shape[0])

# Cache-first: avoids raw .nt access once {schema_label}_ontology_enums.pt exists.
_enum_cache_path = f"{volume}{schema_label}_ontology_enums.pt"
if os.path.exists(_enum_cache_path):
    from types import SimpleNamespace
    _enum_cache = torch.load(_enum_cache_path, map_location='cpu', weights_only=False)
    ontology_obj = SimpleNamespace(
        enum_nodes=_enum_cache['enum_nodes'],
        enum_relations=_enum_cache['enum_relations'],
    )
    print(f"[{schema_label}] Loaded cached enum_nodes/enum_relations from {_enum_cache_path} "
          f"-- no .nt file access needed.")
elif schema_label == 'sulo':
    ontology_obj = Ontology(nt_file_path=sulo_data_path, label_predicate="hasFeature", label_object_pattern=r"SurveyRating/(\d+)")
else:
    ontology_obj = Ontology(nt_file_path=baseline_data_path, label_predicate="hasSurveyRating", label_object_pattern=r"SurveyRating/(\d+)")

## K Sweep Test

In [0]:
# ── EXPERIMENT 2: K-HOP DEPTH SWEEP (RQ1b) ────────────────────────────────────
print("\n>>> STARTING K-SWEEP: STRUCTURAL SENSITIVITY (K = 1, 2, 3)")

# Everything except K (depth/fanout) is held at the already-tuned winning
# config, so any metric difference across rows is attributable to K alone.
def k_sweep_model_factory(hidden_list, bases):
    return RGCNModel(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=hidden_list,
        num_relations=num_relations,
        vector_table=vector_table,
        categorical_row_ids=torch.where(usage_counts > 1)[0],
        num_bases=bases,
    )

# Fan-out fixed per graph at its measured 99th-percentile neighbourhood
# size, held constant across K=1,2,3 to isolate depth alone (unlike the
# GRID_1B_DEPTH schedule above, which varies fan-out per depth instead).
_rq1a_csv_path = f"{volume}graph_geometry/rq1a_structural_comparison.csv"
if not os.path.exists(_rq1a_csv_path):
    raise FileNotFoundError(
        f"Could not find Table 1's structural comparison CSV at: {_rq1a_csv_path}. "
        f"Run extract_graph_geometry.ipynb first -- the K-sweep's fixed fan-out is "
        f"derived from its p99_degree_threshold measurement."
    )
_rq1a_df = pd.read_csv(_rq1a_csv_path, index_col='metric')
_kg_display = {'base': 'Ad-hoc (Base)', 'sulo': 'SULO'}[schema_label]
_p99_fanout = int(round(float(_rq1a_df.loc['p99_degree_threshold', _kg_display])))
print(f">>> Fixed fan-out for {_kg_display}: {_p99_fanout} "
      f"(99th-percentile neighbourhood size, from rq1a_structural_comparison.csv)")

K_SWEEP_VALUES = [
    (1, (_p99_fanout,)),
    (2, (_p99_fanout, _p99_fanout)),
    (3, (_p99_fanout, _p99_fanout, _p99_fanout)),
]

k_sweep_csv_path = f"{volume}rgcn/{schema_label}_k_sweep_results.csv"

k_sweep_df = run_k_sweep(
    model_factory=k_sweep_model_factory,
    graph_data=graph_data, schema_label=schema_label,
    train_indices=train_indices, val_indices=val_indices, test_indices=test_indices,
    vector_table=vector_table, usage_counts=usage_counts,
    hidden_dim=optimal_dim, bases=optimal_bases, lr=optimal_lr, wd=optimal_wd,
    k_values=K_SWEEP_VALUES,
    epochs=50, patience=15,
    csv_path=k_sweep_csv_path,
    resume=True,
)

print(f"\n=======================================================")
print(f" K-SWEEP COMPLETE FOR {schema_label.upper()}")
print(f"=======================================================")
print(k_sweep_df.to_string(index=False))
print(f"=======================================================")